# MR Skeleton Structure Analysis

This notebook analyzes the **structural skeleton** of Metamorphic Relations (MRs), ignoring the specific features and focusing only on the relational patterns.

## Objective

Determine if AI-generated MRs follow the same **structural patterns** as human-designed ones.

## Skeleton Representation

### Numeric MRs (AVs, DCs)

Direction is determined by the **position of the base variable (m1/x1)**:

```
Original: NS(m1) > NS(m2) => TTD(m2) >= TTD(m1) AND D(m2) >= D(m1)

LHS: NS(m1) > NS(m2)      -> m1 has more -> DEC (decreasing from m1 to m2)
RHS: TTD(m2) >= TTD(m1)    -> m2 has more -> INC (increasing from m1 to m2)
     D(m2) >= D(m1)        -> m2 has more -> INC

Skeleton: [DEC] => [INC AND INC]

Tree:
MR
├── LHS
│   └── DEC
└── RHS
    └── AND
        ├── INC
        └── INC
```

### Set/Boolean MRs (DFAs)

```
Original: NFS(x1) == FS(x2) => (R(x1,w1) AND ¬R(x2,w1)) OR (¬R(x1,w1) AND R(x2,w1))

Tree:
MR
├── LHS
│   └── SET_EQ
└── RHS
    └── OR
        ├── AND
        │   ├── PRED
        │   └── NEG_PRED
        └── AND
            ├── NEG_PRED
            └── PRED
```

## Node Types

| Node | Type | Description |
|------|------|-------------|
| INC | Numeric | Increasing from m1/x1 to m2/x2 |
| DEC | Numeric | Decreasing from m1/x1 to m2/x2 |
| EQ | Both | Equality (numeric, set, string, result) |
| INCLUDES | Set | Set inclusion (includesAll) |
| SIZE_LT | Set | Size less than |
| SIZE_GT | Set | Size greater than |
| SIZE_EQ | Set | Size equal |
| PRED | Boolean | Predicate R(x,w), Result(x,w) |
| NEG_PRED | Boolean | Negated predicate ¬R(x,w), ¬Result(x,w) |
| AND | Structural | Conjunction |
| OR | Structural | Disjunction |


In [26]:
# ============================================================
# CONFIGURATION
# ============================================================

INPUT_FILES = [
    "mrs_AVs.txt",
    "mrs_DCs.txt",
    "mrs_DFAs.txt",
    "mrs_Words.txt",
]

OUTPUT_DIR = "output_skeleton"
HEATMAP_FIGSIZE = (14, 12)

print("Configuration loaded")


Configuration loaded


In [2]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}/")


NameError: name 'OUTPUT_DIR' is not defined

In [28]:
# ============================================================
# SKELETON NODE
# ============================================================

@dataclass
class SkeletonNode:
    label: str
    children: List['SkeletonNode'] = field(default_factory=list)

    def __repr__(self):
        if not self.children: return self.label
        return f"{self.label}({', '.join(repr(c) for c in self.children)})"

    def size(self): return 1 + sum(c.size() for c in self.children)

    def to_pattern_string(self):
        if self.label == 'MR':
            lhs = self.children[0].to_pattern_string() if self.children else ''
            rhs = self.children[1].to_pattern_string() if len(self.children) > 1 else ''
            return f'{lhs} => {rhs}'
        elif self.label in ('LHS', 'RHS'):
            return self.children[0].to_pattern_string() if self.children else ''
        elif self.label == 'AND':
            return ' AND '.join(c.to_pattern_string() for c in self.children)
        elif self.label == 'OR':
            parts = [c.to_pattern_string() for c in self.children]
            return ' OR '.join(f'({p})' if ' ' in p else p for p in parts)
        else:
            return self.label

    def pretty(self, indent=0):
        s = '  ' * indent + self.label + '\n'
        for c in self.children: s += c.pretty(indent + 1)
        return s

def print_skeleton_tree(node, prefix='', is_last=True):
    connector = '└── ' if is_last else '├── '
    print(prefix + connector + node.label)
    prefix += '    ' if is_last else '│   '
    for i, child in enumerate(node.children):
        print_skeleton_tree(child, prefix, i == len(node.children) - 1)

print('Skeleton node ready')


Skeleton node ready


In [29]:
# ============================================================
# PARSER
# ============================================================

def detect_mr_style(raw_text):
    if any(kw in raw_text for kw in ['->includesAll', '->size()', '\u00acR(', '\u00acR(x',
                                      '->last()', '->including(', '->remove(',
                                      '\u00acResult(', 'Result(x']):
        return 'set_boolean'
    if re.search(r'R\(x\d', raw_text):
        return 'set_boolean'
    if re.search(r'Result\(x\d', raw_text):
        return 'set_boolean'
    return 'numeric'

# --- Numeric parser (m1/m2 position-aware) ---
BASE_VAR_PATTERN = re.compile(r'(m1|m2|x1|x2)')

def parse_numeric_atom(text):
    text = text.strip()
    op_match = re.search(r'(>=|<=|>|<|==|!=|=)', text)
    if not op_match: raise ValueError(f'No operator in: {text}')
    op = op_match.group(1)
    if op == '=': op = '=='
    if op in ('==', '!='): return SkeletonNode('EQ')
    left = text[:op_match.start()].strip()
    left_vars = BASE_VAR_PATTERN.findall(left)
    base_on_left = any(v in ('m1', 'x1') for v in left_vars)
    if op in ('>', '>='):
        return SkeletonNode('DEC' if base_on_left else 'INC')
    else:
        return SkeletonNode('INC' if base_on_left else 'DEC')

def parse_numeric_side(text):
    text = text.strip()
    parts = re.split(r'\s+AND\s+', text, flags=re.IGNORECASE)
    parts = [p.strip() for p in parts if p.strip()]
    if len(parts) == 0: return SkeletonNode('EMPTY')
    elif len(parts) == 1: return parse_numeric_atom(parts[0])
    else: return SkeletonNode('AND', [parse_numeric_atom(p) for p in parts])

# --- Set/Boolean parser ---
def split_top_level(text, sep_pattern):
    result, last = [], 0
    for m in re.finditer(sep_pattern, text, re.IGNORECASE):
        prefix = text[:m.start()]
        if prefix.count('(') - prefix.count(')') == 0:
            result.append(text[last:m.start()].strip())
            last = m.end()
    result.append(text[last:].strip())
    return [r for r in result if r]

def strip_outer_parens(text):
    text = text.strip()
    if text.startswith('(') and text.endswith(')'):
        depth = 0
        for i, c in enumerate(text):
            if c == '(': depth += 1
            elif c == ')': depth -= 1
            if depth == 0 and i < len(text) - 1: return text
        return text[1:-1].strip()
    return text

def parse_set_boolean_lhs(text):
    text = text.strip()
    parts = split_top_level(text, r'\band\b')
    if len(parts) > 1:
        return SkeletonNode('AND', [parse_set_boolean_lhs(p) for p in parts])
    if '->includesAll' in text: return SkeletonNode('INCLUDES')
    size_vs_size = re.search(r'->size\(\)\s*(>=|<=|>|<|==)\s*.*->size\(\)', text)
    if size_vs_size:
        op = size_vs_size.group(1)
        left_of_op = text[:size_vs_size.start(1)]
        base_on_left = bool(re.search(r'(x1|w1)', left_of_op))
        if op in ('<', '<='):
            return SkeletonNode('SIZE_LT' if base_on_left else 'SIZE_GT')
        elif op in ('>', '>='):
            return SkeletonNode('SIZE_GT' if base_on_left else 'SIZE_LT')
        else: return SkeletonNode('SIZE_EQ')
    size_vs_const = re.search(r'->size\(\)\s*(>=|<=|>|<|==)\s*\d+', text)
    if size_vs_const:
        op = size_vs_const.group(1)
        if op in ('>', '>='): return SkeletonNode('SIZE_GT')
        elif op in ('<', '<='): return SkeletonNode('SIZE_LT')
        else: return SkeletonNode('SIZE_EQ')
    if '==' in text: return SkeletonNode('EQ')
    raise ValueError(f'Cannot parse set/boolean LHS: {text}')

def parse_set_boolean_rhs(text):
    text = text.strip()
    while text.endswith('))') and text.count(')') > text.count('('):
        text = text[:-1]
    parts = split_top_level(text, r'\bor\b')
    if len(parts) > 1:
        return SkeletonNode('OR', [parse_set_boolean_rhs(p) for p in parts])
    parts = split_top_level(text, r'\band\b')
    if len(parts) > 1:
        return SkeletonNode('AND', [parse_set_boolean_rhs(p) for p in parts])
    stripped = strip_outer_parens(text)
    if stripped != text: return parse_set_boolean_rhs(stripped)
    if '->size()' in text:
        if '==' in text: return SkeletonNode('SIZE_EQ')
        elif '<' in text: return SkeletonNode('SIZE_LT')
        elif '>' in text: return SkeletonNode('SIZE_GT')
    if re.match(r'\u00ac\w*\(', text): return SkeletonNode('NEG_PRED')
    if re.match(r'\w+\(x\d', text): return SkeletonNode('PRED')
    raise ValueError(f'Cannot parse set/boolean RHS: {text}')

print('Parser ready')


Parser ready


In [30]:
# ============================================================
# FILE LOADER
# ============================================================

def mr_text_to_skeleton(raw_text):
    clean = raw_text.replace("'=>", "=>").replace("' =>", "=>")
    if '=>' not in clean: raise ValueError(f'No => found in: {raw_text}')
    parts = clean.split('=>', 1)
    left_part, rhs_text = parts[0].strip(), parts[1].strip()
    comma_parts = left_part.split(',')
    if len(comma_parts) >= 4:
        lhs_text = ','.join(comma_parts[3:]).strip()
    elif len(comma_parts) >= 3:
        lhs_text = ','.join(comma_parts[2:]).strip()
    else:
        lhs_text = left_part
    style = detect_mr_style(raw_text)
    if style == 'numeric':
        lhs_node = SkeletonNode('LHS', [parse_numeric_side(lhs_text)])
        rhs_node = SkeletonNode('RHS', [parse_numeric_side(rhs_text)])
    else:
        lhs_node = SkeletonNode('LHS', [parse_set_boolean_lhs(lhs_text)])
        rhs_node = SkeletonNode('RHS', [parse_set_boolean_rhs(rhs_text)])
    return SkeletonNode('MR', [lhs_node, rhs_node])

def parse_mr_line(line):
    line = line.strip()
    if not line or line.startswith('#'): return None
    clean = line.replace("'=>", "=>").replace("' =>", "=>")
    if '=>' not in clean: return None
    left = clean.split('=>', 1)[0].strip()
    cp = left.split(',')
    if len(cp) >= 3:
        group = cp[0].strip()
        p1, p2 = cp[1].strip(), cp[2].strip()
        if p1.startswith('MR'): mr_id, category = p1, p2
        else: category, mr_id = p1, p2
    else:
        group, mr_id, category = 'default', 'MR', 'normal'
    return f'{group}-{category}-{mr_id}', line

def load_mrs_from_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as f: lines = f.readlines()
    skeletons, raw_texts, errors = {}, {}, []
    for line in lines:
        result = parse_mr_line(line)
        if result is None: continue
        full_id, raw_text = result
        if full_id in skeletons: continue
        try:
            skeletons[full_id] = mr_text_to_skeleton(raw_text)
            raw_texts[full_id] = raw_text
        except Exception as e:
            errors.append((full_id, raw_text, str(e)))
    return skeletons, raw_texts, errors

print('File loader ready')


File loader ready


In [31]:
# ============================================================
# SKELETON TREE EDIT DISTANCE (Zhang-Shasha)
# ============================================================

def tree_to_postorder(node):
    result = []
    for child in node.children: result.extend(tree_to_postorder(child))
    result.append(node)
    return result

def leftmost_leaf_index(node, nodes, idx_map):
    if not node.children: return idx_map[id(node)]
    return leftmost_leaf_index(node.children[0], nodes, idx_map)

def skeleton_ted(tree1, tree2):
    nodes1, nodes2 = tree_to_postorder(tree1), tree_to_postorder(tree2)
    n1, n2 = len(nodes1), len(nodes2)
    idx1 = {id(n): i for i, n in enumerate(nodes1)}
    idx2 = {id(n): i for i, n in enumerate(nodes2)}
    l1 = [leftmost_leaf_index(n, nodes1, idx1) for n in nodes1]
    l2 = [leftmost_leaf_index(n, nodes2, idx2) for n in nodes2]
    def get_kr(nodes, l):
        kr = {}
        for i in range(len(nodes)): kr[l[i]] = i
        return sorted(kr.values())
    kr1, kr2 = get_kr(nodes1, l1), get_kr(nodes2, l2)
    td = np.zeros((n1+1, n2+1))
    def cost(a, b):
        if a is None or b is None: return 1
        return 0 if a.label == b.label else 1
    for i in kr1:
        for j in kr2:
            m, n = i-l1[i]+2, j-l2[j]+2
            fd = np.zeros((m, n))
            for x in range(1, m): fd[x, 0] = fd[x-1, 0] + 1
            for y in range(1, n): fd[0, y] = fd[0, y-1] + 1
            for x in range(1, m):
                for y in range(1, n):
                    i1, j1 = l1[i]+x-1, l2[j]+y-1
                    if l1[i1]==l1[i] and l2[j1]==l2[j]:
                        fd[x,y] = min(fd[x-1,y]+1, fd[x,y-1]+1, fd[x-1,y-1]+cost(nodes1[i1],nodes2[j1]))
                        td[i1+1,j1+1] = fd[x,y]
                    else:
                        p, q = l1[i1]-l1[i], l2[j1]-l2[j]
                        fd[x,y] = min(fd[x-1,y]+1, fd[x,y-1]+1, fd[p,q]+td[i1+1,j1+1])
    return int(td[n1, n2])

def compute_skeleton_ted_matrix(skeletons):
    mr_ids = list(skeletons.keys())
    n = len(mr_ids)
    dm = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(i+1, n):
            d = skeleton_ted(skeletons[mr_ids[i]], skeletons[mr_ids[j]])
            dm[i,j] = d; dm[j,i] = d
    return pd.DataFrame(dm, index=mr_ids, columns=mr_ids)

print('Skeleton TED ready')


Skeleton TED ready


In [32]:
# ============================================================
# ANALYSIS
# ============================================================

def analyze_skeleton_patterns(skeletons):
    patterns = []
    for mr_id, sk in skeletons.items():
        patterns.append({'MR_ID': mr_id, 'Pattern': sk.to_pattern_string(), 'Tree_Size': sk.size()})
    return pd.DataFrame(patterns)

def get_pattern_distribution(pattern_df):
    counts = pattern_df['Pattern'].value_counts().reset_index()
    counts.columns = ['Pattern', 'Count']
    counts['Percentage'] = 100 * counts['Count'] / len(pattern_df)
    return counts

def analyze_ted_matrix(dist_matrix, name):
    n = len(dist_matrix)
    distances = dist_matrix.values[np.triu_indices(n, k=1)]
    lines = []
    lines.append('=' * 60)
    lines.append(f'SKELETON TED ANALYSIS: {name}')
    lines.append('=' * 60)
    lines.append(f'\nTotal pairs: {len(distances)}')
    lines.append(f'Mean distance: {np.mean(distances):.2f}')
    lines.append(f'Median: {np.median(distances):.2f} | Min: {np.min(distances):.0f} | Max: {np.max(distances):.0f}')
    lines.append(f'Std: {np.std(distances):.2f}')
    lines.append('\n--- Distance Distribution ---')
    for d in sorted(set(distances)):
        count = np.sum(distances == d)
        lines.append(f'  TED = {d:.0f}: {count:3d} pairs ({100*count/len(distances):5.1f}%)')
    identical = [(dist_matrix.index[i], dist_matrix.columns[j])
                 for i in range(n) for j in range(i+1, n) if dist_matrix.iloc[i,j]==0]
    lines.append(f'\nIdentical structure pairs (TED=0): {len(identical)}')
    if identical:
        for a, b in identical: lines.append(f'  {a} <-> {b}')
    for l in lines: print(l)
    return {'mean_dist': np.mean(distances), 'n_identical': len(identical),
            'identical_pairs': identical, 'report_lines': lines}

print('Analysis ready')


Analysis ready


In [33]:
# ============================================================
# PDF GENERATION
# ============================================================

from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

DISTANCE_COLORS = ['#F0F0F0','#C6DBEF','#9ECAE1','#6BAED6','#3182BD','#08519C','#063E78','#042F5A','#021E3C']
CLUSTER_COLORS = ['#E41A1C','#377EB8','#4DAF4A','#984EA3','#FF7F00','#A65628','#F781BF','#66C2A5','#FC8D62','#8DA0CB','#E78AC3','#A6D854']

def generate_distance_heatmap_pdf(dist_matrix, output_path, title):
    matrix = dist_matrix.values.astype(int)
    max_dist = int(matrix.max())
    n_colors = min(max_dist + 1, len(DISTANCE_COLORS))
    colors = DISTANCE_COLORS[:n_colors]
    cmap = mcolors.ListedColormap(colors)
    norm = mcolors.BoundaryNorm(list(range(n_colors + 1)), cmap.N)

    condensed = squareform(matrix.astype(float))
    Z = linkage(condensed, method='average')
    unique_dists = sorted(np.unique(matrix))
    n_clusters = min(len(unique_dists), len(CLUSTER_COLORS))
    clusters = fcluster(Z, t=n_clusters, criterion='maxclust')
    cluster_map = {i+1: CLUSTER_COLORS[i] for i in range(n_clusters)}
    row_colors = pd.Series(clusters, index=dist_matrix.index).map(cluster_map)

    g = sns.clustermap(
        dist_matrix, method='average', metric='precomputed',
        row_linkage=Z, col_linkage=Z,
        cmap=cmap, norm=norm, linewidths=0.5, linecolor='white',
        figsize=HEATMAP_FIGSIZE, annot=True, fmt='d',
        annot_kws={'size': max(6, min(10, 120//len(matrix))), 'weight': 'bold'},
        row_colors=row_colors, col_colors=row_colors,
        dendrogram_ratio=(0.12, 0.12), cbar_pos=None,
        tree_kws={'linewidths': 1.5},
    )

    for text in g.ax_heatmap.texts:
        val = int(text.get_text())
        cidx = min(val, len(colors)-1)
        rgb = mcolors.to_rgb(colors[cidx])
        text.set_color('white' if 0.299*rgb[0]+0.587*rgb[1]+0.114*rgb[2] < 0.5 else '#333333')

    g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=8)
    g.fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)

    legend_elements = []
    for d in unique_dists:
        legend_elements.append(Patch(facecolor=colors[min(int(d),len(colors)-1)], edgecolor='gray', label=f'Distance {int(d)}'))
    legend_elements.append(Patch(facecolor='none', edgecolor='none', label=''))
    for c in range(1, n_clusters+1):
        legend_elements.append(Patch(facecolor=CLUSTER_COLORS[c-1], edgecolor='gray', label=f'Cluster {c}'))
    g.ax_heatmap.legend(handles=legend_elements, loc='center left',
                         bbox_to_anchor=(1.15, 0.5), frameon=True, fontsize=8,
                         title='Legend', title_fontsize=9)
    plt.savefig(output_path, format='pdf', dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {output_path}')

def generate_pattern_distribution_pdf(pattern_dist, output_path, title):
    fig, ax = plt.subplots(figsize=(12, max(6, len(pattern_dist)*0.5)))
    y_pos = range(len(pattern_dist))
    ax.barh(y_pos, pattern_dist['Count'], color='steelblue', alpha=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(pattern_dist['Pattern'], fontsize=9)
    ax.set_xlabel('Count')
    ax.set_title(f'Skeleton Pattern Distribution\n{title}', fontweight='bold')
    for i, (count, pct) in enumerate(zip(pattern_dist['Count'], pattern_dist['Percentage'])):
        ax.text(count+0.1, i, f'{count} ({pct:.1f}%)', va='center', fontsize=9)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(output_path, format='pdf', dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {output_path}')

print('PDF generators ready')


PDF generators ready


In [34]:
# ============================================================
# REPORT
# ============================================================

def generate_skeleton_report(filename, skeletons, raw_texts, pattern_dist, analysis, output_path):
    lines = ['='*70, f'SKELETON STRUCTURE ANALYSIS REPORT: {filename}', '='*70, '',
             'Direction determined by base variable (m1/x1) position.', '']
    lines += ['-'*50, 'SUMMARY', '-'*50]
    lines.append(f'Total MRs: {len(skeletons)}')
    lines.append(f'Unique patterns: {len(pattern_dist)}')
    lines.append(f'Identical pairs: {analysis["n_identical"]}')
    lines.append(f'Mean TED: {analysis["mean_dist"]:.2f}')
    lines += ['', '-'*50, 'PATTERN DISTRIBUTION', '-'*50]
    lines.append(pattern_dist.to_string(index=False))
    lines += ['', '-'*50, 'MR SKELETON DETAILS', '-'*50]
    for mr_id, sk in skeletons.items():
        lines.append(f'\n{mr_id}:')
        lines.append(f'  Pattern: {sk.to_pattern_string()}')
        for tl in sk.pretty().strip().split('\n'): lines.append(f'    {tl}')
    lines += ['']
    lines.extend(analysis['report_lines'])
    lines += ['\n' + '='*70, 'END OF REPORT', '='*70]
    with open(output_path, 'w', encoding='utf-8') as f: f.write('\n'.join(lines))
    print(f'  Saved: {output_path}')

print('Report generator ready')


Report generator ready


In [35]:
# ============================================================
# MAIN PROCESSING
# ============================================================

def process_mr_file(filepath, output_dir):
    filename = Path(filepath).stem
    print(f'\n{"="*60}')
    print(f'Processing: {filepath}')
    print(f'{"="*60}')

    skeletons, raw_texts, errors = load_mrs_from_file(filepath)
    print(f'MRs loaded: {len(skeletons)}')
    if errors:
        print(f'Parse errors: {len(errors)}')
        for fid, raw, err in errors: print(f'  {fid}: {err}')

    if len(skeletons) == 0: return {'filename': filename, 'status': 'empty'}

    print('\n--- Skeleton Examples ---')
    for mr_id, sk in list(skeletons.items())[:3]:
        print(f'\n{mr_id}:')
        print(f'  Pattern: {sk.to_pattern_string()}')
        print_skeleton_tree(sk)

    pattern_df = analyze_skeleton_patterns(skeletons)
    pattern_dist = get_pattern_distribution(pattern_df)
    print(f'\nUnique patterns: {len(pattern_dist)}')

    print('\nComputing Skeleton TED...')
    dist_matrix = compute_skeleton_ted_matrix(skeletons)
    print('')
    analysis = analyze_ted_matrix(dist_matrix, filename)

    print('\nGenerating outputs...')
    generate_distance_heatmap_pdf(dist_matrix,
        os.path.join(output_dir, f'{filename}_skeleton_ted_distance.pdf'),
        f'Skeleton TED Distance\n{filename}')
    generate_pattern_distribution_pdf(pattern_dist,
        os.path.join(output_dir, f'{filename}_skeleton_patterns.pdf'), filename)
    generate_skeleton_report(filename, skeletons, raw_texts, pattern_dist, analysis,
        os.path.join(output_dir, f'{filename}_skeleton_report.txt'))
    dist_matrix.to_csv(os.path.join(output_dir, f'{filename}_skeleton_ted_matrix.csv'))
    pattern_df.to_csv(os.path.join(output_dir, f'{filename}_skeleton_patterns.csv'), index=False)

    print(f'\nResult: {len(skeletons)} MRs -> {len(pattern_dist)} unique patterns')
    return {'filename': filename, 'n_mrs': len(skeletons), 'n_patterns': len(pattern_dist),
            'n_identical_pairs': analysis['n_identical'], 'mean_ted': analysis['mean_dist'], 'status': 'success'}

print('Main processing ready')


Main processing ready


In [36]:
# ============================================================
# EXECUTION
# ============================================================

print('=' * 60)
print('MR SKELETON STRUCTURE ANALYSIS')
print('=' * 60)

results = []
for fp in INPUT_FILES:
    if os.path.exists(fp):
        results.append(process_mr_file(fp, OUTPUT_DIR))
    else:
        print(f'\nFile not found: {fp}')

print('\n' + '=' * 60)
print('SUMMARY')
print('=' * 60)
for r in results:
    if r.get('status') == 'success':
        print(f'\n{r["filename"]}:')
        print(f'  MRs: {r["n_mrs"]}')
        print(f'  Unique skeleton patterns: {r["n_patterns"]}')
        print(f'  Identical structure pairs: {r["n_identical_pairs"]}')
        print(f'  Mean Skeleton TED: {r["mean_ted"]:.2f}')


MR SKELETON STRUCTURE ANALYSIS

Processing: mrs_AVs.txt
MRs loaded: 17

--- Skeleton Examples ---

base-normal-MR1:
  Pattern: DEC => INC AND INC
└── MR
    ├── LHS
    │   └── DEC
    └── RHS
        └── AND
            ├── INC
            └── INC

base-normal-MR2:
  Pattern: DEC => DEC AND DEC
└── MR
    ├── LHS
    │   └── DEC
    └── RHS
        └── AND
            ├── DEC
            └── DEC

base-normal-MR3:
  Pattern: DEC => DEC AND DEC
└── MR
    ├── LHS
    │   └── DEC
    └── RHS
        └── AND
            ├── DEC
            └── DEC

Unique patterns: 6

Computing Skeleton TED...

SKELETON TED ANALYSIS: mrs_AVs

Total pairs: 136
Mean distance: 1.94
Median: 2.00 | Min: 0 | Max: 4
Std: 1.14

--- Distance Distribution ---
  TED = 0:  25 pairs ( 18.4%)
  TED = 1:  10 pairs (  7.4%)
  TED = 2:  57 pairs ( 41.9%)
  TED = 3:  36 pairs ( 26.5%)
  TED = 4:   8 pairs (  5.9%)

Identical structure pairs (TED=0): 25
  base-normal-MR1 <-> base-noIF-MR4
  base-normal-MR1 <-> base-noIF-MR6